In [1]:
!pip install -q fastapi uvicorn pydantic sentence-transformers spacy tensorflow requests
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os
os.makedirs("app", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)

upload the models in artifacts and app.py in app folder

In [3]:
import threading
import time
import subprocess
import re
import uvicorn

def run_server():
    uvicorn.run("app.api:app", host="127.0.0.1", port=8000, log_level="info")

# Start FastAPI in background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(5)  # Wait for SBERT & Keras models to load

# Download and start Cloudflare Quick Tunnel (no auth token required)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

tunnel = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in tunnel.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print(f"\nAPI Live at: {public_url}")
        print(f"Interactive Docs: {public_url}/docs\n")
        break


API Live at: https://electric-one-obligation-confident.trycloudflare.com
Interactive Docs: https://electric-one-obligation-confident.trycloudflare.com/docs

